In [1]:
import sys

import polars as pl
from polars import selectors as cs

sys.path.append("/app")

from src.polars_utils import normalize, random_split

In [2]:
setA = pl.scan_csv(
    "/mnt/data/raw/sepsis/training_setA/training/*.psv",
    separator="|",
    null_values="NaN",
    include_file_paths="file_path",
)

setB = pl.scan_csv(
    "/mnt/data/raw/sepsis/training_setB/training_setB/*.psv",
    separator="|",
    null_values="NaN",
    include_file_paths="file_path",
)

df = pl.concat([setA, setB], how="vertical_relaxed")

df = df.cast(
    {
        cs.by_index(range(35)): pl.Float32,
        "^Unit.$": pl.Int32,
        "Gender": pl.Boolean,
        "HospAdmTime": pl.Float32,
        "SepsisLabel": pl.Boolean,
    }
)

df = df.with_columns(
    pl.col("file_path")
    .str.split("/")
    .list.last()
    .str.extract(r"p([[:digit:]]+)\.psv")
    .cast(pl.Int32)
    .alias("patient_id"),
    cs.float().fill_null(float("nan")),
    cs.integer().fill_null(-1),
)

df = df.with_columns(
    pl.col("^Unit.$") + 1,
    normalize(pl.col("HospAdmTime", "Age")),
    time=pl.duration(
        hours=pl.col("ICULOS") - pl.col("ICULOS").first().over("patient_id")
    ),
)

df = df.set_sorted("patient_id").drop("file_path")
# Right now, filter(*) doesn't work for LazyFrame agg
# see https://github.com/pola-rs/polars/issues/18733
columns = set(df.collect_schema().keys())
static_columns = {"Age", "Gender", "Unit1", "Unit2", "HospAdmTime"}
list_cols = columns - static_columns - {"patient_id", "SepsisLabel"}
df = (
    df.group_by("patient_id")
    .agg(
        pl.col(list_cols).filter(pl.col("ICULOS") < 72),
        pl.col(static_columns).first(),
        pl.col("SepsisLabel").any(),
    )
    .drop("ICULOS")
)

df = random_split(df, train=0.7, val=0.15, test=0.15)

In [3]:
temporal_norm = df.select(pl.col("time").list.max().median()).collect()
df = (
    df.filter(
        pl.col("time").list.len().is_between(5, 1000),
        (pl.col("time").list.max() / temporal_norm).is_between(0.1, 10),
    )
    .with_columns(
        pl.col("time").list.eval(pl.element() / temporal_norm),
        target="SepsisLabel",
        balance_col="SepsisLabel",
    )
    .drop("SepsisLabel")
)
print(temporal_norm)

shape: (1, 1)
┌──────────────┐
│ time         │
│ ---          │
│ duration[μs] │
╞══════════════╡
│ 1d 13h       │
└──────────────┘


In [ ]:
df = df.drop("patient_id").collect()

In [ ]:
df.write_parquet("/mnt/data/preprocessed/sepsis.parquet")